In [11]:
import pandas as pd
import numpy as np
import os
import yaml
import json

# claude --resume 1066c47a-c343-4f01-89fa-4817a778ca26

In [12]:
# L1_ORDER = {"Background": 0, "White Matter": 1, "Gray Matter": 2, "Cerebellum White Matter": 3, "Cerebellum Gray Matter": 4, 
#        "Deep Gray Matter": 5, "Brainstem": 6, "CSF": 7}


# ontology_v0 = pd.read_csv("data/inputs/nextbrain_ontology_v1.csv")
# ontology_v0_l1 = ontology_v0["L1_tissue"].unique()
# unique_ids = ontology_v0["allen_id"].unique()
# unique_ids = [int(i) for i in unique_ids]

# print("L1 tissue in ontology_v0:", ontology_v0_l1)

# NAMES_MAPPING = {
#     "Background": "Background",
#     "White Matter": "Cerebral WM",
#     "Gray Matter": "Cerebral GM",
#     "Cerebellum White Matter": "Cerebellar WM",
#     "Cerebellum Gray Matter": "Cerebellar GM",
#     "Deep Gray Matter": "Deep GM",
#     "Brainstem": "Brainstem",
#     "CSF": "CSF / ventricles"
# }


# mapping_dict = {}
# for key, value in NAMES_MAPPING.items():
#     ids_list = ontology_v0[ontology_v0["L1_tissue"] == value]["allen_id"].to_list()
#     # convert to int
#     ids_list = [int(i) for i in ids_list]
#     mapping_dict[key] = ids_list

# mapped_ids = [id for sublist in mapping_dict.values() for id in sublist]

# # print missing IDs in unique_ids that are not in mapped_ids
# missing_ids = [id for id in unique_ids if id not in mapped_ids]
# print("Missing IDs in mapping_dict:", missing_ids)


# # Structures that are Other in the ontology 
# mapping_dict["Brainstem"].append(506)  # Brainstem

# # Node structures like the whole cortex
# mapping_dict["Gray Matter"].append(3)  # Brainstem

# # print the number of IDs in the mapping_dict
# total_ids = sum([len(value) for value in mapping_dict.values()])
# print(f"Matched ids: {total_ids} = {len(ontology_v0['allen_id'].unique())} (mapped ids vs unique ids in ontology_v1) if 3 is added left side should differ by 1")


# # print("Mapping dict:", mapping_dict)
# mapping_l1_path = "data/outputs/l1_names_mapping.yaml"
# # save as yaml filliwing the order of the keys in L1_ORDER
# with open(mapping_l1_path, "w") as f:
#     yaml.dump(mapping_dict, f, sort_keys=False)

### FInd incoherences with converted_16l.yaml

In [15]:
l16_path = "data/inputs/GMM_label_grouping.yaml"
lsupersynth_path = "data/inputs/NextBrainToSuperSynth.yaml"
removed_labels_path = "data/inputs/labels_to_remove.json"

# load the GMM_label_grouping.yaml and flatten the list of single-key dicts into one dict
with open(l16_path, "r") as f:
    raw = yaml.safe_load(f)
l16 = {k: [int(i) for i in v] for d in raw["Combined Structures"] for k, v in d.items()}

with open(lsupersynth_path, "r") as f:
    raw = yaml.safe_load(f)
lsupersynth = raw

with open(removed_labels_path) as f:
    labels_to_remove = json.load(f)
removed_ids = [int(i) for i in labels_to_remove]

l16_ids = [id for sublist in l16.values() for id in sublist]
lsupersynth_ids = [id for sublist in lsupersynth.values() for id in sublist]


print("Structures in l16:", l16.keys())
print("Structures in lsupersynth:", lsupersynth.keys())


# print("Structures in l16 but not in lsupersynth:", set(l16) - set(lsupersynth))
# print("Structures in lsupersynth but not in l16:", set(lsupersynth) - set(l16))

print("IDs in l16 but not in lsupersynth:", set(l16_ids) - set(lsupersynth_ids))
print("IDs in lsupersynth but not in l16:", set(lsupersynth_ids) - set(l16_ids))

print("Unique IDs in l16:", len(set(l16_ids)))
print("Unique IDs in lsupersynth:", len(set(lsupersynth_ids)))
print("Total unique IDs in l16 and lsupersynth:", len(set(l16_ids + lsupersynth_ids)))


print("removed ids in l16:", set(removed_ids) & set(l16_ids))
print("removed ids in lsupersynth:", len(set(removed_ids) & set(lsupersynth_ids)), set(removed_ids) & set(lsupersynth_ids), )
# # id -> structure lookup for both dictionaries
# id_to_mapping = {i: k for k, ids in mapping_dict.items() for i in ids}
# id_to_converted = {i: k for k, ids in converted_16l.items() for i in ids}

# # structures present in only one of the dictionaries
# print("Structures only in mapping_dict:", set(mapping_dict) - set(converted_16l))
# print("Structures only in converted_16l:", set(converted_16l) - set(mapping_dict))

# # per-structure differences
# for key in L1_ORDER:
#     m = set(mapping_dict.get(key, []))
#     c = set(converted_16l.get(key, []))
#     if m == c:
#         print(f"\n[OK] {key}: {len(m)} ids match")
#         continue
#     print(f"\n[MISMATCH] {key}: mapping_dict={len(m)} ids, converted_16l={len(c)} ids")
#     only_m = sorted(m - c)
#     only_c = sorted(c - m)
#     if only_m:
#         print("  only in mapping_dict :", [(i, f"converted_16l -> {id_to_converted.get(i, 'MISSING')}") for i in only_m])
#     if only_c:
#         print("  only in converted_16l:", [(i, f"mapping_dict -> {id_to_mapping.get(i, 'MISSING')}") for i in only_c])

Structures in l16: dict_keys(['Background', 'White Matter', 'Gray Matter', 'Cerebellum White Matter', 'Cerebellum Gray Matter', 'Caudate and Accumbens', 'Putamen', 'White Thalamus', 'Gray Thalamus', 'Pallidum', 'Substantia Nigra and STN and Red Nucleus', 'Gray Brainstem nuclei', 'White Brainstem nuclei', 'Gray hypothalamus', 'White hypothalamus', 'Cerebellar Other', 'Hippocampal White Matter'])
Structures in lsupersynth: dict_keys(['Cerebral-White-Matter', 'Cerebral-Cortex', 'Cerebellum-White-Matter', 'Cerebellum-Cortex', 'Thalamus', 'Caudate', 'Putamen', 'Pallidum', 'Hippocampus', 'Amygdala', 'Accumbens-area', 'Claustrum', 'Brain-Stem', 'Optic-Chiasm', 'HypoThal-noMB', 'Fornix', 'MammillaryBody', 'Basal-Forebrain', 'SeptalNuc'])
IDs in l16 but not in lsupersynth: {0, 50}
IDs in lsupersynth but not in l16: {515, 771, 647, 649, 779, 527, 528, 655, 659, 660, 669, 543, 415, 546, 677, 682, 683, 689, 690, 693, 694, 695, 703, 581, 589, 717, 591, 592, 724, 725, 726, 599, 598, 600, 731, 610, 6